# 01 - Clean Salon Appointment Log

**Input:** data/salon/appoitments_anaoymized.csv (raw, never modified)  

**Output:** data/processed/salon_clean.csv  

Steps follow Task 2 sections D4 and D5a.  

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

# Notebook lives in notebooks/, so go up one folder to reach data/
RAW_PATH = "../data/salon/appointments_anonymized.csv"

df = pd.read_csv(RAW_PATH)

# Stop the notebook if the raw file is not what D4 describes
assert df.shape == (186, 12), f"Unexpected shape: {df.shape}"

print(df.shape)
print(list(df.columns))
df.head()

(186, 12)
['appointment_hash', 'client_pid', 'provider_pid', 'services', 'booking_method', 'status', 'deposit_amount', 'deposit_collected', 'charge_hash', 'appt_datetime_utc', 'booked_datetime_utc', 'card_exp_year']


,appointment_hash,client_pid,provider_pid,services,booking_method,status,deposit_amount,deposit_collected,charge_hash,appt_datetime_utc,booked_datetime_utc,card_exp_year
0,e65a69f03395cf1714976bcded41418571003014a676fd...,Client_105,Provider_001,Women's Haircut (Add on deep conditioning ),Self-Scheduled,cancelled,0,False,NaN,2025-02-13T10:00:00Z,2025-02-11T18:38:00Z,2028.0
1,a1df894d96675bdd23df9a701ffcb0b3aac44f6ad674c3...,Client_064,Provider_001,Women's Haircut,Self-Scheduled,completed,0,False,1eaaf197db40adee2a37bb689a09a6cd126fab9091916b...,2025-02-27T09:00:00Z,2025-02-05T12:55:00Z,NaN
2,fe5da58adef34c224e34699e0ae91dad58ff4c310d95bf...,Client_022,Provider_001,All over color,Self-Scheduled,completed,0,False,8518a65435c1bbeb016e1d2fe36ab217045a9cfb42cd8b...,2025-03-17T15:00:00Z,2025-03-03T11:26:00Z,NaN
3,16640665331a36c8b45a35b27aa846b71050c1e829be01...,Client_079,Provider_001,Balayage/lived in/Foilayage,Self-Scheduled,completed,0,False,5eb22f2454c8a1cf335e75eee280bc9a574fadb10d9b2a...,2025-03-26T14:30:00Z,2025-03-07T17:50:00Z,NaN
4,b2b4633d7294fb0785bf4cf899f0071c478157dfb2dee1...,Client_084,Provider_001,"Partial Custom Highlight, Add on base color",Self-Scheduled,completed,0,False,df9f604d703e0268a8f86885eb0c64f6b3639f1187514c...,2025-03-27T12:00:00Z,2025-03-09T12:01:00Z,NaN


In [2]:
# D4 says every deposit is zero - here shows it before dropping
print("deposit_amount values:\n", df["deposit_amount"].value_counts(dropna=False))

# Find both deposit columns by name instead of guessing
deposit_cols = [c for c in df.columns if c.startswith("deposit")]
print("Deposit columns found:", deposit_cols)

# D4/ D5a: payment fields and Client ID are not used in analysis
drop_cols = ["charge_hash", "card_exp_year", "client_pid"] + deposit_cols
df = df.drop(columns=drop_cols)

print("Columns remaining:", df.shape[1])
print(list(df.columns))

deposit_amount values:
 deposit_amount
0    186
Name: count, dtype: int64
Deposit columns found: ['deposit_amount', 'deposit_collected']
Columns remaining: 7
['appointment_hash', 'provider_pid', 'services', 'booking_method', 'status', 'appt_datetime_utc', 'booked_datetime_utc']


In [4]:
# D4 validity check: every appointmnet should be unique
n_dupes = df["appointment_hash"].duplicated().sum()
print("Duplicate appointment_hash values:", n_dupes)
assert n_dupes == 0, "Duplicates found - investigate before continuing"

# D5a: hash is kept only for duplicate check, then removed
df = df.drop(columns=["appointment_hash"])
print("Columns remaining:", df.shape[1])

Duplicate appointment_hash values: 0
Columns remaining: 6


In [6]:
# utc=True reads the timestamps as UTC (as labeled in file)
# tz_localize(None) then removes the label without shifting hour,
# so 14:00 stays 14:00. D4: times are treated as local salon time
df["appt_dt"] = pd.to_datetime(df["appt_datetime_utc"], utc=True).dt.tz_localize(None)
df["booked_dt"] = pd.to_datetime(df["booked_datetime_utc"], utc=True).dt.tz_localize(None)

# Raw text columns removed as not needed
df = df.drop(columns=["appt_datetime_utc", "booked_datetime_utc"])

# CHeck against profiling: 182 in 9 - 17, one at 8, two at 18 and one at 1
print(df["appt_dt"].dt.hour.value_counts().sort_index())

appt_dt
1      1
8      1
9     30
10    23
11    24
12    24
13    18
14    18
15    16
16    19
17    10
18     2
Name: count, dtype: int64


In [8]:
# Hour of the appointment - basis for demand tiers (C1)
df["appt_hour"] = df["appt_dt"].dt.hour

# Days between booking and appointment (lead time)
df["lead_days"] = (df["appt_dt"] - df["booked_dt"]).dt.days

# Week number counted from the first appointment in the log.
# Built from the data (not calendar weeks) due to per-client
# date shift (D4). //Using whole num division
first_appt = df["appt_dt"].min()
df["week_index"] = (df["appt_dt"] - first_appt).dt.days // 7

# Check against D4
print("median lead days:", (df["lead_days"] < 0).sum()) # expecting 14
print("Negative lead times:", df["lead_days"].median()) # expecting 2
print("week_index range:", df["week_index"].min(), "to", df["week_index"].max()) # expecting 0 - 43

df.head()

median lead days: 2
Negative lead times: 14.0
week_index range: 0 to 43


,provider_pid,services,booking_method,status,appt_dt,booked_dt,appt_hour,lead_days,week_index
0,Provider_001,Women's Haircut (Add on deep conditioning ),Self-Scheduled,cancelled,2025-02-13 10:00:00,2025-02-11 18:38:00,10,1,0
1,Provider_001,Women's Haircut,Self-Scheduled,completed,2025-02-27 09:00:00,2025-02-05 12:55:00,9,21,1
2,Provider_001,All over color,Self-Scheduled,completed,2025-03-17 15:00:00,2025-03-03 11:26:00,15,14,4
3,Provider_001,Balayage/lived in/Foilayage,Self-Scheduled,completed,2025-03-26 14:30:00,2025-03-07 17:50:00,14,18,5
4,Provider_001,"Partial Custom Highlight, Add on base color",Self-Scheduled,completed,2025-03-27 12:00:00,2025-03-09 12:01:00,12,17,6
